# In-context learning (few-shot prompting)

_Few-shot & Transfer Learning_

**Show a Large Language Model a few worked examples in the prompt, and it does the task on the spot — no training, no weight changes.**

A Large Language Model (LLM) is a big neural network trained to predict the next word.

       In-context learning is a surprise: you can make the LLM do a brand-new task just by typing a few worked examples into the prompt. The prompt is called the context.

---

This notebook is a practice scaffold for the **In-context learning (few-shot prompting)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Visualize the data & results

_As we add more in-context demonstrations per class, how does accuracy on a held-out query set rise and then plateau?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier

# REAL data: 1797 handwritten 8x8 digit images, pixel features scaled 0..1.
# Real LLMs can't run here, so this is a k-NN proxy: "K demonstrations per class
# placed in the prompt" becomes "a k-NN given K real examples per class".
digits = load_digits()
X = digits.data / 16.0
y = digits.target
classes = np.unique(y)

# Fixed held-out query (test) set; the rest is the demonstration pool.
X_pool, X_test, y_pool, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

rng = np.random.default_rng(0)
by_class = {c: np.where(y_pool == c)[0] for c in classes}

Ks = [0, 1, 2, 5, 10, 20]      # in-context demonstrations PER CLASS
accs = []
for K in Ks:
    if K == 0:
        # zero-shot: no demonstrations -> can only guess the most-frequent class
        acc = DummyClassifier(strategy="most_frequent").fit(X_pool, y_pool).score(X_test, y_test)
    else:
        trials = []
        for _ in range(20):            # average over random example draws (choice matters)
            idx = []
            for c in classes:
                pool_c = by_class[c]
                idx += rng.choice(pool_c, size=min(K, len(pool_c)), replace=False).tolist()
            idx = np.array(idx)
            knn = KNeighborsClassifier(n_neighbors=min(5, len(idx))).fit(X_pool[idx], y_pool[idx])
            trials.append(knn.score(X_test, y_test))
        acc = float(np.mean(trials))
    accs.append(round(acc, 3))

print(list(zip(Ks, accs)))
# -> [(0, 0.102), (1, 0.213), (2, 0.566), (5, 0.794), (10, 0.879), (20, 0.93)]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You give an LLM three labelled examples in the prompt, then a new sentence to classify. What is $K$, and is this zero-shot, one-shot, or few-shot?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count the demonstration pairs placed before the query. — _$K$ is just the number of worked examples in the context — here there are three._
- Match $K$ to the naming rule: $K = 0$ is zero-shot, $K = 1$ is one-shot, $K$ > 1 is few-shot. — _Three examples is more than one, so it falls in the few-shot range._

**Answer:** $K = 3$, so this is few-shot prompting. The model conditions on all three pairs in a single forward pass; its weights never change.

</details>

**Problem 2.** A teammate says in-context learning "fine-tunes the model on the prompt examples". Why is that wrong?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what fine-tuning does: it runs gradient descent and updates the model's weights. — _Fine-tuning, like [fs-transfer-learning] and [fs-meta-learning], physically changes the network's numbers._
- Recall what in-context learning does: it only places examples in the input text and runs one forward pass. — _No gradients are computed and no weight is updated — the model is frozen._

**Answer:** It is wrong because in-context learning changes no weights. The examples only condition a single frozen forward pass; nothing is trained. That is the sharp contrast with [fs-transfer-learning] and [fs-meta-learning], which do update weights by gradient descent.

</details>